# Exploración y preparación de los Conteos Censales de Participación Ciudadana 2024

**Proyecto:** Hackea la participación: Datos que construyen democracia  
**Reto 1:** Perfiles ciudadanos de participación  
**Fuente principal:** Instituto Nacional Electoral (INE), Conteos Censales de Participación Ciudadana 2024  
**Archivo de trabajo:** `DatosAbiertos_DECEyEC_ConteosCensales2024.zip`

## Objetivo del notebook

Integrar, revisar y preparar la base de los Conteos Censales de Participación Ciudadana 2024 para construir posteriormente perfiles de participación mediante técnicas de segmentación.

En esta etapa se realiza:

- inventario de los archivos proporcionados por el INE;
- lectura y validación de una entidad como prueba;
- integración de las 32 entidades federativas;
- revisión de variables y controles de consistencia;
- agrupación de edades;
- construcción de una unidad de análisis socio-territorial;
- cálculo de indicadores de participación, abstención y no especificación;
- identificación de grupos con pocos registros determinados;
- exportación de una base procesada lista para integrarse con los demás años.

> **Nota metodológica:** este notebook corresponde únicamente a la preparación de 2024. El clustering no se realiza todavía. Antes de segmentar, se deberá integrar 2009, 2012, 2015, 2018 y 2021.


## Fuente, alcance y limitaciones de los datos

El archivo de descripción incluido por el INE señala que la fuente de los conteos son los **cuadernillos de la lista nominal** de las casillas electorales aprobadas por los consejos distritales.

Por ello, es importante considerar desde esta etapa que:

- los registros **no corresponden a microdatos personales**; son conteos agregados;
- no se contempla la participación de personas que votaron en **casillas especiales**;
- tampoco se contempla la participación de mexicanas y mexicanos **residentes en el extranjero**;
- la sección se asocia al lugar de residencia consignado en la credencial electoral;
- los resultados pueden diferir de otras fuentes, como los cómputos distritales, debido a que la fuente de información es distinta.

Estas limitaciones deben mencionarse en el documento técnico final para evitar interpretar los indicadores obtenidos como equivalentes a todos los cómputos oficiales de la elección.


In [ ]:
import pandas as pd
import zipfile

### Nota de reproducibilidad

El notebook está pensado para ejecutarse en Google Colab. El archivo ZIP de los datos de 2024 debe cargarse en `/content/` antes de ejecutar las siguientes celdas.  
Si Colab reinicia el entorno de ejecución, será necesario volver a subir el ZIP.


In [ ]:
ruta_zip = "/content/DatosAbiertos_DECEyEC_ConteosCensales2024.zip"

In [ ]:
with zipfile.ZipFile(ruta_zip, "r") as zip_ref:
    archivos = zip_ref.namelist()

for archivo in archivos:
    print(archivo)

DatosAbiertos_DECEyEC_ConteosCensales2024/
DatosAbiertos_DECEyEC_ConteosCensales2024/01_descripcioncampos_conteoscensales2024.txt
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_ags.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_bc.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_bcs.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_camp.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_cdmx.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_chih.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_chis.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_coah.csv
DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_col.csv
DatosAbiertos_DECEyEC_ConteosCensales2

In [ ]:
archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

print("Número de archivos CSV:", len(archivos_csv))

Número de archivos CSV: 32


In [ ]:
archivos_csv[:5]


['DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_ags.csv',
 'DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_bc.csv',
 'DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_bcs.csv',
 'DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_camp.csv',
 'DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_cdmx.csv']

### Resultado del inventario

Se identificaron **32 archivos CSV**, uno por cada entidad federativa, además del archivo de descripción de campos proporcionado por el INE.

Esta separación por entidad hace necesario integrar los archivos en una sola base nacional antes de realizar el análisis comparativo.


In [ ]:
archivo_ags = [
    archivo for archivo in archivos_csv
    if archivo.endswith("_ags.csv")
][0]

archivo_ags

'DatosAbiertos_DECEyEC_ConteosCensales2024/datosabiertos_deceyec_conteoscensales2024_ags.csv'

In [ ]:
with zipfile.ZipFile(ruta_zip, "r") as zip_ref:
    with zip_ref.open(archivo_ags) as archivo:
        df_ags = pd.read_csv(archivo)

df_ags.head()

,AELEC,FELECCION,EDOCVE,EDONOM,MPIOCVE,MPIONOM,SECCION,TIPOSEC,DEF,DEL,SEXO,EDAD,LN,SV,NV,NS
0,2024,06/02/2024,1,AGUASCALIENTES,5,Jesús María,413,M,1,NaN,0,21,133,85,48,0
1,2024,06/02/2024,1,AGUASCALIENTES,5,Jesús María,413,M,1,NaN,0,57,77,65,12,0
2,2024,06/02/2024,1,AGUASCALIENTES,7,Rincón de Romos,437,M,1,1.0,0,23,36,14,22,0
3,2024,06/02/2024,1,AGUASCALIENTES,7,Rincón de Romos,450,M,1,1.0,0,37,14,8,6,0
4,2024,06/02/2024,1,AGUASCALIENTES,7,Rincón de Romos,450,M,1,1.0,0,40,15,7,8,0


In [ ]:
df_ags.shape

(91826, 16)

In [ ]:
df_ags.columns

Index(['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM',
       'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV',
       'NS'],
      dtype='object')

In [ ]:
df_ags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91826 entries, 0 to 91825
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   AELEC      91826 non-null  int64  
 1   FELECCION  91826 non-null  object 
 2   EDOCVE     91826 non-null  int64  
 3   EDONOM     91826 non-null  object 
 4   MPIOCVE    91826 non-null  int64  
 5   MPIONOM    91826 non-null  object 
 6   SECCION    91826 non-null  int64  
 7   TIPOSEC    91826 non-null  object 
 8   DEF        91826 non-null  int64  
 9   DEL        91211 non-null  float64
 10  SEXO       91826 non-null  int64  
 11  EDAD       91826 non-null  int64  
 12  LN         91826 non-null  int64  
 13  SV         91826 non-null  int64  
 14  NV         91826 non-null  int64  
 15  NS         91826 non-null  int64  
dtypes: float64(1), int64(11), object(4)
memory usage: 11.2+ MB


## Diccionario de variables relevante para el análisis

De acuerdo con el archivo de descripción incluido en el ZIP:

| Variable | Significado | Uso en esta etapa |
|---|---|---|
| `AELEC` | Año de elección | Control del año |
| `FELECCION` | Fecha de la jornada electoral | Referencia |
| `EDOCVE` | Clave INE de entidad | Identificador |
| `EDONOM` | Nombre de la entidad | Caracterización territorial |
| `MPIOCVE` | Clave INE de municipio | Identificador territorial |
| `MPIONOM` | Nombre del municipio | Referencia territorial |
| `SECCION` | Sección electoral | Nivel original de agregación |
| `TIPOSEC` | Tipo de sección: U, M o R | Caracterización territorial |
| `DEF` | Distrito Electoral Federal | No se utiliza en la primera agregación |
| `DEL` | Distrito Electoral Local | No se utiliza en la primera agregación |
| `SEXO` | 0 = Hombres, 1 = Mujeres, 2 = No binario | Caracterización sociodemográfica |
| `EDAD` | Edad cumplida al día de la elección | Construcción de grupos de edad |
| `LN` | Lista nominal | Tamaño del grupo |
| `SV` | Registros con marca de voto | Participación determinada |
| `NV` | Registros sin marca de voto | Abstención determinada |
| `NS` | Registros con participación no especificada | Control de calidad/cobertura |

`NS` incluye casos en los que el cuadernillo no tenía marcas de votación, no estaba disponible para captura o alguna página no estaba disponible.


## Integración de las 32 entidades federativas

En esta sección se integran los archivos CSV correspondientes a las
32 entidades federativas en una sola base de datos nacional.

In [ ]:
dataframes = []

with zipfile.ZipFile(ruta_zip, "r") as zip_ref:
    for archivo in archivos_csv:
        with zip_ref.open(archivo) as archivo_csv:
            df = pd.read_csv(archivo_csv)

        dataframes.append(df)

df_2024 = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_2024.shape
df_2024.head()
df_2024.tail()

,AELEC,FELECCION,EDOCVE,EDONOM,MPIOCVE,MPIONOM,SECCION,TIPOSEC,DEF,DEL,SEXO,EDAD,LN,SV,NV,NS
9773725,2024,06/02/2024,32,ZACATECAS,38,Pinos,1143,U,4,14.0,0,46,7,2,5,0
9773726,2024,06/02/2024,32,ZACATECAS,38,Pinos,1121,U,4,14.0,0,54,10,7,3,0
9773727,2024,06/02/2024,32,ZACATECAS,38,Pinos,1121,U,4,14.0,0,76,3,0,3,0
9773728,2024,06/02/2024,32,ZACATECAS,38,Pinos,1130,U,4,14.0,0,80,4,3,1,0
9773729,2024,06/02/2024,32,ZACATECAS,38,Pinos,1143,U,4,14.0,0,84,1,0,1,0


In [ ]:
df_2024["EDONOM"].nunique()

32

In [ ]:
df_2024["EDONOM"].unique()

array(['AGUASCALIENTES', 'BAJA CALIFORNIA', 'BAJA CALIFORNIA SUR',
       'CAMPECHE', 'CIUDAD DE MEXICO', 'CHIHUAHUA', 'CHIAPAS', 'COAHUILA',
       'COLIMA', 'DURANGO', 'GUERRERO', 'GUANAJUATO', 'HIDALGO',
       'JALISCO', 'MEXICO', 'MICHOACAN', 'MORELOS', 'NAYARIT',
       'NUEVO LEON', 'OAXACA', 'PUEBLA', 'QUERETARO', 'QUINTANA ROO',
       'SINALOA', 'SAN LUIS POTOSI', 'SONORA', 'TABASCO', 'TAMAULIPAS',
       'TLAXCALA', 'VERACRUZ', 'YUCATAN', 'ZACATECAS'], dtype=object)

In [ ]:
df_2024["EDONOM"].value_counts()

,count
EDONOM,
MEXICO,951310
CIUDAD DE MEXICO,808977
VERACRUZ,699401
JALISCO,539890
SINALOA,473681
GUANAJUATO,457616
CHIHUAHUA,418127
PUEBLA,409591
MICHOACAN,392150


In [ ]:
df_2024.shape

(9773730, 16)

In [ ]:
df_2024["EDONOM"].nunique()

32

In [ ]:
df_2024[["LN", "SV", "NV", "NS"]].isnull().sum()

,0
LN,0
SV,0
NV,0
NS,0


### Control de integridad posterior a la integración

La base nacional contiene **9,773,730 filas y 16 variables**, e incluye las **32 entidades federativas**.

Además, `LN`, `SV`, `NV` y `NS` no presentan valores nulos. Estas cuatro variables son centrales porque permiten reconstruir el tamaño del grupo y distinguir participación determinada, abstención determinada y registros no especificados.

> Una fila de la base original **no representa a una persona individual**. En 2024, los registros están desagregados por territorio, sexo y edad y contienen conteos asociados a esas combinaciones.


## Revisión y preparación de variables

Antes de agregar la información, se revisan los valores disponibles
para edad, sexo y tipo de sección, así como la consistencia entre
lista nominal, sí votaron, no votaron y registros no especificados.

In [ ]:
print("Edad mínima:", df_2024["EDAD"].min())
print("Edad máxima:", df_2024["EDAD"].max())

print("\nValores de SEXO:")
print(df_2024["SEXO"].value_counts().sort_index())

print("\nValores de TIPOSEC:")
print(df_2024["TIPOSEC"].value_counts())

Edad mínima: 18
Edad máxima: 140

Valores de SEXO:
SEXO
0    4817367
1    4956258
2        105
Name: count, dtype: int64

Valores de TIPOSEC:
TIPOSEC
U    6368734
R    2483182
M     921814
Name: count, dtype: int64


### Observaciones sobre edad, sexo y tipo de sección

- La edad observada va de **18 a 140 años**. El valor máximo se conserva tal como aparece en la fuente; posteriormente queda incluido en el grupo `75+`. No se corrige ni elimina sin una justificación adicional.
- La categoría `SEXO = 2` (No binario) aparece con una frecuencia muy reducida: **105 registros en la base original**. Por esta razón, su comportamiento debe revisarse con cuidado al construir perfiles longitudinales.
- Los tres tipos de sección presentes son: `U` = Urbana, `M` = Mixta y `R` = Rural.

La baja frecuencia de algunas categorías no implica que deban eliminarse. Primero se conservarán y posteriormente se evaluará su efecto sobre la segmentación.


In [ ]:
consistencia = (
    df_2024["LN"] ==
    df_2024["SV"] + df_2024["NV"] + df_2024["NS"]
)

print("Registros consistentes:", consistencia.sum())
print("Registros inconsistentes:", (~consistencia).sum())
print("Porcentaje consistente:", consistencia.mean() * 100)

Registros consistentes: 9773730
Registros inconsistentes: 0
Porcentaje consistente: 100.0


### Verificación de consistencia interna

Se comprobó en las **9,773,730 filas** que:

$$
LN = SV + NV + NS
$$

La igualdad se cumple en el **100 % de los registros**. Esto confirma que las tres categorías (`SV`, `NV` y `NS`) reconstruyen completamente la lista nominal reportada en cada fila.

Esta validación es importante antes de cualquier agregación, ya que permite comprobar posteriormente que no se pierdan registros al transformar la base.


### Agrupación de edades

Para facilitar la interpretación de los perfiles ciudadanos, las edades
individuales se agrupan en intervalos quinquenales, conservando un grupo
específico para personas de 18 a 19 años y agrupando las edades de 75 años
o más en una sola categoría.


**Decisión metodológica provisional.** La edad se resume en grupos principalmente quinquenales para facilitar la interpretación de los perfiles. Se mantienen dos excepciones:

- `18-19`, por ser el primer grupo de edad electoral;
- `75+`, que concentra todas las edades de 75 años en adelante.

La agrupación puede revisarse más adelante si la comparación histórica 2009–2024 muestra que otra definición produce mayor comparabilidad o estabilidad.


In [ ]:
bins = [
    18, 20, 25, 30, 35, 40, 45,
    50, 55, 60, 65, 70, 75, float("inf")
]

labels = [
    "18-19",
    "20-24",
    "25-29",
    "30-34",
    "35-39",
    "40-44",
    "45-49",
    "50-54",
    "55-59",
    "60-64",
    "65-69",
    "70-74",
    "75+"
]

df_2024["GRUPO_EDAD"] = pd.cut(
    df_2024["EDAD"],
    bins=bins,
    labels=labels,
    right=False
)

In [ ]:
df_2024["GRUPO_EDAD"].value_counts().sort_index()

,count
GRUPO_EDAD,
18-19,274166
20-24,696462
25-29,698158
30-34,696798
35-39,695103
40-44,694112
45-49,694588
50-54,694031
55-59,690491


In [ ]:
df_2024["GRUPO_EDAD"].isnull().sum()

np.int64(0)

In [ ]:
df_2024["SEXO_DESC"] = df_2024["SEXO"].map({
    0: "Hombre",
    1: "Mujer",
    2: "No binario"
})

df_2024["TIPOSEC_DESC"] = df_2024["TIPOSEC"].map({
    "U": "Urbana",
    "M": "Mixta",
    "R": "Rural"
})

In [ ]:
df_2024[
    ["EDONOM", "EDAD", "GRUPO_EDAD", "SEXO_DESC", "TIPOSEC_DESC", "LN", "SV", "NV", "NS"]
].head(10)

,EDONOM,EDAD,GRUPO_EDAD,SEXO_DESC,TIPOSEC_DESC,LN,SV,NV,NS
0,AGUASCALIENTES,21,20-24,Hombre,Mixta,133,85,48,0
1,AGUASCALIENTES,57,55-59,Hombre,Mixta,77,65,12,0
2,AGUASCALIENTES,23,20-24,Hombre,Mixta,36,14,22,0
3,AGUASCALIENTES,37,35-39,Hombre,Mixta,14,8,6,0
4,AGUASCALIENTES,40,40-44,Hombre,Mixta,15,7,8,0
5,AGUASCALIENTES,54,50-54,Hombre,Mixta,18,11,7,0
6,AGUASCALIENTES,59,55-59,Hombre,Mixta,8,7,1,0
7,AGUASCALIENTES,78,75+,Hombre,Mixta,3,2,1,0
8,AGUASCALIENTES,80,75+,Hombre,Mixta,1,1,0,0
9,AGUASCALIENTES,69,65-69,Hombre,Mixta,22,16,1,5


## Construcción de la unidad de análisis

Para esta primera versión se define como unidad de análisis la combinación:

$$
\text{Entidad} + \text{Grupo de edad} + \text{Sexo} + \text{Tipo de sección}
$$

Por ejemplo:

`AGUASCALIENTES | 20-24 | Mujer | Urbana`

La intención es pasar de millones de filas territoriales muy específicas a grupos socio-territoriales comparables e interpretables.

> **Importante para el clustering posterior:** estas variables describen e identifican a los grupos, pero no necesariamente todas deben entrar como variables numéricas del algoritmo. Una estrategia que conviene evaluar será formar los clusters principalmente con el **historial de participación** y utilizar edad, sexo, entidad y tipo de sección para caracterizarlos después. También puede compararse contra una especificación que incorpore variables sociodemográficas codificadas, siempre justificando su peso.


In [ ]:
df_2024_agrupado = (
    df_2024
    .groupby(
        ["EDONOM", "GRUPO_EDAD", "SEXO_DESC", "TIPOSEC_DESC"],
        observed=True,
        as_index=False
    )[["LN", "SV", "NV", "NS"]]
    .sum()
)

In [ ]:
df_2024_agrupado.shape

(2546, 8)

### Resultado de la agregación

La base se redujo de **9,773,730 registros originales** a **2,546 grupos socio-territoriales observados**.

Se utilizó `observed=True`, por lo que solo se conservan combinaciones de categorías que realmente existen en los datos. La ausencia de una combinación teórica no se interpreta automáticamente como un dato faltante.


In [ ]:
denominador = (
    df_2024_agrupado["SV"] +
    df_2024_agrupado["NV"]
)

df_2024_agrupado["PARTICIPACION_PCT"] = (
    df_2024_agrupado["SV"] / denominador
) * 100

df_2024_agrupado["ABSTENCION_PCT"] = (
    df_2024_agrupado["NV"] / denominador
) * 100

df_2024_agrupado.loc[
    denominador == 0,
    ["PARTICIPACION_PCT", "ABSTENCION_PCT"]
] = pd.NA

### Definición de los indicadores

En este notebook la participación y la abstención se calculan únicamente sobre registros con resultado determinado:

$$
N_{determinado} = SV + NV
$$

$$
Participación(\%) = \frac{SV}{SV+NV}\times100
$$

$$
Abstención(\%) = \frac{NV}{SV+NV}\times100
$$

Los registros `NS` **no forman parte del denominador** de estos dos porcentajes. Por lo tanto, participación + abstención suma 100 % por construcción, aunque existan registros no especificados.

Para no perder de vista esa información, se incorpora también el porcentaje de no especificados respecto a la lista nominal:


In [ ]:
df_2024_agrupado["NO_ESPECIFICADO_PCT"] = (
    df_2024_agrupado["NS"] / df_2024_agrupado["LN"]
) * 100

df_2024_agrupado.loc[
    df_2024_agrupado["LN"] == 0,
    "NO_ESPECIFICADO_PCT"
] = pd.NA


In [ ]:
df_2024_agrupado.head(20)

,EDONOM,GRUPO_EDAD,SEXO_DESC,TIPOSEC_DESC,LN,SV,NV,NS,PARTICIPACION_PCT,ABSTENCION_PCT,NO_ESPECIFICADO_PCT
0,AGUASCALIENTES,18-19,Hombre,Mixta,5702,2855,2585,262,52.481618,47.518382,4.594879
1,AGUASCALIENTES,18-19,Hombre,Rural,2588,1300,1239,49,51.201260,48.798740,1.893354
2,AGUASCALIENTES,18-19,Hombre,Urbana,14908,7288,7297,323,49.969146,50.030854,2.166622
3,AGUASCALIENTES,18-19,Mujer,Mixta,5797,3345,2192,260,60.411775,39.588225,4.485078
4,AGUASCALIENTES,18-19,Mujer,Rural,2760,1756,959,45,64.677716,35.322284,1.630435
5,AGUASCALIENTES,18-19,Mujer,Urbana,15130,8198,6593,339,55.425597,44.574403,2.240582
6,AGUASCALIENTES,20-24,Hombre,Mixta,15988,6448,8801,739,42.284740,57.715260,4.622217
7,AGUASCALIENTES,20-24,Hombre,Rural,8242,3051,5041,150,37.703905,62.296095,1.819947
8,AGUASCALIENTES,20-24,Hombre,Urbana,45163,18879,25225,1059,42.805641,57.194359,2.344840
9,AGUASCALIENTES,20-24,Mujer,Mixta,15890,8397,6776,717,55.341725,44.658275,4.512272


In [ ]:
print("Totales base original:")
print(df_2024[["LN", "SV", "NV", "NS"]].sum())

print("\nTotales base agrupada:")
print(df_2024_agrupado[["LN", "SV", "NV", "NS"]].sum())

Totales base original:
LN    98330348
SV    54899228
NV    37105466
NS     6325654
dtype: int64

Totales base agrupada:
LN    98330348
SV    54899228
NV    37105466
NS     6325654
dtype: int64


In [ ]:
df_2024[["LN", "SV", "NV", "NS"]].sum().equals(
    df_2024_agrupado[["LN", "SV", "NV", "NS"]].sum()
)

True

### Conservación de totales

La prueba devuelve `True`: la agregación conserva exactamente los totales de `LN`, `SV`, `NV` y `NS`.

Totales observados en la base 2024:

- **Lista nominal (`LN`)**: 98,330,348
- **Sí votaron (`SV`)**: 54,899,228
- **No votaron (`NV`)**: 37,105,466
- **No especificado (`NS`)**: 6,325,654

A partir de estos conteos, la participación entre registros con resultado determinado es aproximadamente **59.67 %**. Este valor debe interpretarse dentro del alcance de los Conteos Censales descrito al inicio y no como sustituto de otros indicadores oficiales basados en fuentes distintas.


In [ ]:
totales_2024 = df_2024_agrupado[["LN", "SV", "NV", "NS"]].sum()
n_determinado_2024 = totales_2024["SV"] + totales_2024["NV"]

resumen_nacional_2024 = pd.Series({
    "LN": totales_2024["LN"],
    "SV": totales_2024["SV"],
    "NV": totales_2024["NV"],
    "NS": totales_2024["NS"],
    "N_DETERMINADO": n_determinado_2024,
    "PARTICIPACION_DETERMINADA_PCT":
        totales_2024["SV"] / n_determinado_2024 * 100,
    "ABSTENCION_DETERMINADA_PCT":
        totales_2024["NV"] / n_determinado_2024 * 100,
    "NO_ESPECIFICADO_SOBRE_LN_PCT":
        totales_2024["NS"] / totales_2024["LN"] * 100
})

resumen_nacional_2024


,0
LN,9.833035e+07
SV,5.489923e+07
NV,3.710547e+07
NS,6.325654e+06
N_DETERMINADO,9.200469e+07
PARTICIPACION_DETERMINADA_PCT,5.967003e+01
ABSTENCION_DETERMINADA_PCT,4.032997e+01
NO_ESPECIFICADO_SOBRE_LN_PCT,6.433064e+00


In [ ]:
df_2024_agrupado[
    ["PARTICIPACION_PCT", "ABSTENCION_PCT"]
].describe()

,PARTICIPACION_PCT,ABSTENCION_PCT
count,2546.000000,2546.000000
mean,61.629867,38.370133
std,13.769244,13.769244
min,0.000000,0.000000
25%,52.354486,28.727397
50%,62.633784,37.366216
75%,71.272603,47.645514
max,100.000000,100.000000


In [ ]:
(
    df_2024_agrupado["PARTICIPACION_PCT"] +
    df_2024_agrupado["ABSTENCION_PCT"]
).describe()

,0
count,2.546000e+03
mean,1.000000e+02
std,3.199419e-15
min,1.000000e+02
25%,1.000000e+02
50%,1.000000e+02
75%,1.000000e+02
max,1.000000e+02


### Precaución al interpretar el promedio de los grupos

El `describe()` anterior muestra una media de participación de aproximadamente **61.63 %**, pero **ese número no es la participación nacional**.

Es un promedio simple de los 2,546 grupos: un grupo con pocas personas pesa exactamente lo mismo que uno con decenas de miles. Para obtener un indicador agregado se deben utilizar los conteos (`SV` y `NV`) como ponderadores, como se hizo en el resumen nacional anterior.

Asimismo, `PARTICIPACION_PCT` y `ABSTENCION_PCT` contienen la misma información en sentido inverso porque:

$$
Participación + Abstención = 100
$$

Por ello, **no deben incluirse simultáneamente como variables del futuro K-Means**, ya que se duplicaría la misma dimensión de información.


In [ ]:
print("Grupos con LN < 10:",
      (df_2024_agrupado["LN"] < 10).sum())

print("Grupos con LN < 30:",
      (df_2024_agrupado["LN"] < 30).sum())

print("Grupos con LN < 100:",
      (df_2024_agrupado["LN"] < 100).sum())

Grupos con LN < 10: 48
Grupos con LN < 30: 52
Grupos con LN < 100: 61


In [ ]:
df_2024_agrupado[
    df_2024_agrupado["LN"] < 30
].sort_values("LN").head(30)

,EDONOM,GRUPO_EDAD,SEXO_DESC,TIPOSEC_DESC,LN,SV,NV,NS,PARTICIPACION_PCT,ABSTENCION_PCT,NO_ESPECIFICADO_PCT
31,AGUASCALIENTES,35-39,No binario,Urbana,1,1,0,0,100.0,0.0,0.0
92,BAJA CALIFORNIA,20-24,No binario,Mixta,1,1,0,0,100.0,0.0,0.0
93,BAJA CALIFORNIA,20-24,No binario,Urbana,1,0,1,0,0.0,100.0,0.0
100,BAJA CALIFORNIA,25-29,No binario,Urbana,1,1,0,0,100.0,0.0,0.0
107,BAJA CALIFORNIA,30-34,No binario,Urbana,1,1,0,0,100.0,0.0,0.0
422,CHIHUAHUA,30-34,No binario,Urbana,1,1,0,0,100.0,0.0,0.0
490,CIUDAD DE MEXICO,20-24,No binario,Mixta,1,1,0,0,100.0,0.0,0.0
815,GUANAJUATO,25-29,No binario,Urbana,1,1,0,0,100.0,0.0,0.0
822,GUANAJUATO,30-34,No binario,Urbana,1,1,0,0,100.0,0.0,0.0
1075,JALISCO,40-44,No binario,Urbana,1,1,0,0,100.0,0.0,0.0


### Grupos de tamaño muy reducido

Las tasas extremas (0 % o 100 %) pueden aparecer legítimamente cuando un grupo tiene muy pocos registros. Por ejemplo, 1 voto de 1 registro determinado produce 100 %, pero ese porcentaje es mucho menos estable que una tasa calculada con miles de registros.

Por esta razón, el tamaño relevante para evaluar estabilidad no es únicamente `LN`, sino:

$$
N_{determinado}=SV+NV
$$

A continuación se crea una bandera provisional para identificar grupos con menos de 100 registros determinados. **La bandera no elimina observaciones**; solo permitirá realizar un análisis de sensibilidad más adelante.


In [ ]:
df_2024_agrupado["N_DETERMINADO"] = (
    df_2024_agrupado["SV"] +
    df_2024_agrupado["NV"]
)

In [ ]:
df_2024_agrupado["GRUPO_PEQUENO"] = (
    df_2024_agrupado["N_DETERMINADO"] < 100
)

In [ ]:
df_2024_agrupado["GRUPO_PEQUENO"].value_counts()

,count
GRUPO_PEQUENO,
False,2485
True,61


In [ ]:
pd.crosstab(
    df_2024_agrupado["SEXO_DESC"],
    df_2024_agrupado["GRUPO_PEQUENO"],
    margins=True
)


GRUPO_PEQUENO,False,True,All
SEXO_DESC,,,
Hombre,1242,6,1248
Mujer,1243,5,1248
No binario,0,50,50
All,2485,61,2546


### Interpretación de la bandera `GRUPO_PEQUENO`

Se identifican **61 grupos** con menos de 100 registros determinados y **2,485 grupos** que superan o igualan ese umbral.

La categoría No binario es especialmente dispersa: los datos 2024 contienen muy pocos registros en comparación con Hombres y Mujeres. Esto deberá considerarse al homologar los años anteriores, pues no se debe asumir que la categoría existe con la misma definición y cobertura en todos los procesos electorales.

El umbral de 100 es **provisional**, no una regla estadística definitiva. Antes del clustering se recomienda comparar al menos:

1. resultados usando todos los grupos;
2. resultados excluyendo temporalmente los grupos marcados como pequeños.

Si la estructura de los clusters es estable en ambos casos, se obtiene evidencia de robustez. Si cambia sustancialmente, la decisión deberá discutirse en el documento técnico.


### Validación de la base agregada

Se verificó que la agregación conserva la totalidad de los registros de
lista nominal, participación, abstención y casos no especificados de la
base original.

Asimismo, se identificaron grupos con tamaños poblacionales reducidos.
Estos registros se conservan en la base y se incorpora una variable de
control para evaluar posteriormente su efecto sobre el método de
segmentación.

## Decisiones metodológicas que deben conservarse en la documentación

| Decisión | Justificación actual | Estado |
|---|---|---|
| Integrar las 32 entidades | Construir una base nacional comparable | Adoptada |
| Agrupar edades | Mejorar interpretación y reducir granularidad | Provisional |
| Unidad: entidad × edad × sexo × tipo de sección | Crear grupos socio-territoriales interpretables | Provisional |
| Participación = SV/(SV+NV) | Separar casos con resultado determinado de `NS` | Adoptada para este notebook |
| Conservar `NS` | Evitar ocultar problemas de cobertura/captura | Adoptada |
| No eliminar grupos pequeños | Evitar pérdida prematura de información | Adoptada |
| Marcar `N_DETERMINADO < 100` | Preparar análisis de sensibilidad | Provisional |
| No usar participación y abstención juntas en clustering | Son perfectamente redundantes | Adoptada |
| Usar variables sociodemográficas principalmente para caracterizar perfiles | Evitar que categorías territoriales dominen el clustering | A validar con datos 2009–2024 |

### Pendientes antes del clustering

1. Homologar la estructura de 2009, 2012, 2015, 2018 y 2021.
2. Verificar la comparabilidad histórica de `SEXO`, especialmente la categoría No binario.
3. Verificar que los grupos de edad puedan construirse de la misma manera en todos los años.
4. Revisar cambios históricos en `TIPOSEC` y nombres/claves territoriales.
5. Construir para cada grupo su historial de participación por elección.
6. Definir variables de trayectoria, por ejemplo: promedio histórico, tendencia y variabilidad.
7. Estandarizar las variables numéricas antes de aplicar K-Means.
8. Evitar usar `LN`, `SV` o `NV` como características directas si no se desea que el tamaño poblacional domine la distancia; el tamaño del grupo puede evaluarse como **peso muestral** o en análisis de sensibilidad.
9. Comparar resultados con y sin grupos pequeños y, si procede, probar ponderación por `N_DETERMINADO`.
10. Revisar el porcentaje `NS` para evitar perfiles sustentados en grupos con alta proporción de participación no especificada.


## Exportación de la base procesada

Antes de exportar, se recomienda ejecutar el notebook **desde la primera celda hasta esta sección** para asegurar que las columnas añadidas durante la documentación —incluido `NO_ESPECIFICADO_PCT`— queden incorporadas en el CSV final.

El archivo generado debe almacenarse en el repositorio en:

`datos/procesados/conteos_censales_2024_agrupado.csv`


In [ ]:
df_2024_agrupado.to_csv(
    "/content/conteos_censales_2024_agrupado.csv",
    index=False,
    encoding="utf-8-sig"
)

# Resumen de esta etapa para el documento técnico

Se integraron los 32 archivos estatales de los Conteos Censales de Participación Ciudadana 2024, obteniendo una base nacional de **9,773,730 registros**. Se verificó la presencia de las 32 entidades y la ausencia de valores nulos en las variables principales de conteo (`LN`, `SV`, `NV` y `NS`). Además, se comprobó que en el 100 % de los registros se cumple la relación `LN = SV + NV + NS`.

Para reducir la granularidad de la información y construir unidades interpretables, las edades se agruparon en 13 categorías y posteriormente los datos se agregaron por **entidad, grupo de edad, sexo y tipo de sección**, produciendo **2,546 grupos socio-territoriales**. La agregación conserva exactamente los totales de la base original.

La tasa de participación se calculó sobre los registros con condición de voto determinada (`SV + NV`), manteniendo por separado los registros `NS`. También se creó una variable de control para identificar grupos con menos de 100 registros determinados; estos grupos no se eliminaron, ya que su efecto será evaluado mediante análisis de sensibilidad en la etapa de segmentación.

## Próximo paso

Procesar y homologar la base **2021** con la misma lógica y comparar su estructura con 2024. Después se repetirá el procedimiento para 2018, 2015, 2012 y 2009, con el objetivo de construir un historial de participación 2009–2024 antes de aplicar el método de clustering.


## Tabla de compatibilidad para la integración histórica
Esta tabla registra qué variables del esquema común están disponibles en 2024.
Más adelante se combinará con las revisiones de 2009, 2012, 2015, 2018 y 2021.

In [ ]:
variables_2024 = [
    "EDONOM",
    "EDAD",
    "SEXO",
    "TIPOSEC",
    "LN",
    "SV",
    "NV",
    "NS"
]

compatibilidad_2024 = pd.DataFrame({
    "ANIO": 2024,
    "VARIABLE": variables_2024,
    "PRESENTE_2024": [
        variable in df_2024.columns
        for variable in variables_2024
    ],
    "OBSERVACION": ""
})

compatibilidad_2024

,ANIO,VARIABLE,PRESENTE_2024,OBSERVACION
0,2024,EDONOM,True,
1,2024,EDAD,True,
2,2024,SEXO,True,
3,2024,TIPOSEC,True,
4,2024,LN,True,
5,2024,SV,True,
6,2024,NV,True,
7,2024,NS,True,


In [ ]:
compatibilidad_2024.to_csv(
    "/content/compatibilidad_variables_2024.csv",
    index=False,
    encoding="utf-8-sig"
)